In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import itertools
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, describe, mode, fisher_exact
import scipy as sp
import scipy.stats as stats
# from scipy.spatial.distance import jaccard, euclidean, cosine
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot
from wordcloud import WordCloud
from sklearn.decomposition import NMF, non_negative_factorization
from collections import Counter
from scipy.stats import hypergeom
from scipy.stats.mstats import gmean
from scipy.cluster import hierarchy
import squarify
try:
    import piplite
    await piplite.install(['ipywidgets'])
except ImportError:
    pass
import ipywidgets as widgets
import json

today = date.today()

**combining datasets and reducing precision**

In [59]:
datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
DHS_per_f = datadir + 'regions_25kb_CT20230214.DHS.factornormNMF.sum_enrich_corank_percentile.hdf5'
epi_per_f = datadir + 'regions_25kb_CT20230214.epilogos.statesum_rank_percentile.hdf5'

# load region percentiles
DHS_per, DHS_factors, DHS_color = load_per(DHS_per_f)
epi_per, epi_factors, epi_color = load_per(epi_per_f)

In [60]:
percents = np.hstack((DHS_per, epi_per))
factor_order = np.append(DHS_factors, epi_factors)
metric_order = np.append(np.array(['DHSs'] * len(DHS_factors)),
                         np.array(['chromatin states'] *
                                  len(epi_factors)))
colors = np.append(DHS_color, epi_color)

In [61]:
percents.shape

(88036, 34)

In [62]:
out_f = datadir + 'regions_25kb_CT20230321.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.npy'
print(out_f)
np.save(out_f, percents.astype('float32'))

/home/ctrader/AG/NarrativeGenomics/region_narration/data/regions_25kb_CT20230321.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.npy


**optimizing functions**

loading data - hdf5

In [71]:
t = time.time()

datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
DHS_per_f = datadir + 'regions_25kb_CT20230214.DHS.factornormNMF.sum_enrich_corank_percentile.hdf5'
epi_per_f = datadir + 'regions_25kb_CT20230214.epilogos.statesum_rank_percentile.hdf5'

# load region percentiles
DHS_per, DHS_factors, DHS_color = load_per(DHS_per_f)
epi_per, epi_factors, epi_color = load_per(epi_per_f)

# join data from each metric
join_data_per_metric(DHS_per, DHS_factors, DHS_color, epi_per, epi_factors, epi_color)

print(time.time() - t)

0.9674258232116699


loading data - compressed tsv

In [72]:
t = time.time()

datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
per_f = datadir + 'regions_25kb_CT20230321.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.npy'

percents = np.load(per_f)

print(time.time() - t)

0.009004831314086914


loading regions - old

In [134]:
t = time.time()
regions_path = datadir + 'hg38.25kb.maxMean.CT20230118.sorted.bed'

f = open(regions_path, 'r')
regions = []
regions_sep = []
for line in f.readlines():
    line = line.strip().split('\t')
    regions_sep.append(line)
    regions.append(line[0] + ':' + line[1] + '-' + line[2])
f.close()
regions_sep = np.array(regions_sep)

print(time.time() - t)

0.3315892219543457


In [126]:
# out_f = datadir + 'hg38.25kb.maxMean.CT20230118.sorted.bed.npy'
# print(out_f)
# np.save(out_f, regions_sep)

/home/ctrader/AG/NarrativeGenomics/region_narration/data/hg38.25kb.maxMean.CT20230118.sorted.bed.npy


loading regions - new

In [143]:
t = time.time()
regions_path = datadir + 'hg38.25kb.maxMean.CT20230118.sorted.bed.npy'

regions_sep = np.load(regions_path)

print(time.time() - t)

0.010492324829101562


**Old functions**

In [ ]:
def join_data_per_metric(DHS_per, DHS_factors, DHS_color, epi_per, epi_factors, epi_color):
    global percents, factor_order, metric_order, colors
    percents = np.hstack((DHS_per, epi_per))
    factor_order = np.append(DHS_factors, epi_factors)
    metric_order = np.append(np.array(['DHSs'] * len(DHS_factors)),
                             np.array(['chromatin states'] *
                                      len(epi_factors)))
    colors = np.append(DHS_color, epi_color)
    
    
def load_per(per_f):
    hdf5_f = h5py.File(per_f, 'r')
    percentile = hdf5_f['percentile'][:]
    factors = np.char.decode(hdf5_f['factors'][:])
    color = np.char.decode(hdf5_f['colors'][:])
    hdf5_f.close()
    return percentile, factors, color

**Final**

In [79]:
def roiURL(roi, top_regions, detail_level, target_factors_str,
           link_type='dual'):
    today = date.today()
    roiurl_name = ('region_narration_simsearch_regions_roi_' + str(roi) +
                   '_detail_level_' + str(detail_level) + '_factors_' +
                   target_factors_str + '_scale_25kb_CT' +
                   today.strftime("%Y%m%d") + '.bed')
    bed_path_name = os.path.join('/home/ctrader/public_html/', roiurl_name)
    np.savetxt(bed_path_name, top_regions, delimiter='\t', fmt='%s')
    link_path1 = ('https://epilogos.altius.org:1234/?application=viewer&' +
                  'sampleSet=vC&mode=single&genome=hg38&model=18&complex' +
                  'ity=KL&group=all&chrLeft=chr1&chrRight=chr1&start=800' +
                  '0&stop=35000&roiURL=https%3A%2F%2Fencode%3Acollabor8%' +
                  '40resources.altius.org%2F~ctrader%2F' + roiurl_name)
    link_path2 = ('https://index.altius.org:1234/?application=viewer&mod' +
                  'e=pd&chrLeft=chr11&chrRight=chr11&start=47329800&stop' +
                  '=47356800&roiURL=https%3A%2F%2Fencode%3Acollabor8%40r' +
                  'esources.altius.org%2F~ctrader%2F' + roiurl_name)
    link_path_dual = ('https://index.altius.org:3000/?application=viewer' +
                      '&mode=cf&&roiURL=https%3A%2F%2Fencode%3Acollabor8' +
                      '%40resources.altius.org%2F~ctrader%2F' + roiurl_name)
    if link_type == 'dual':
        return link_path_dual
    else:
        return link_path1, link_path2
    

def top_region_sig(percents, top_regions):
    # mean and std of factor percentiles in all regions
    all_mean = np.mean(percents, axis=0)
    all_std = np.std(percents, axis=0)

    # mean of factor percentiles in most similar regions
    top_region_mean = np.mean(percents[top_regions[:, 1:]], axis=1)

    # Z score
    Z = (top_region_mean - all_mean) / (all_std / np.sqrt(top_regions.shape[1] - 1))

    # p-value (one sided)
    pval = stats.norm.sf(Z)

    return pval, Z


def simsearch(percents, roi, included_factors):

    # factors the query region rank highest in
    query_factor_order = np.argsort(-percents[roi])

    # factors to search over based on detail level
    target_factors = query_factor_order[
        np.isin(query_factor_order, included_factors)
    ]

    # convolution
    roi_percents = percents[roi, target_factors]
    roi_conv_all = percents[:, target_factors] * roi_percents

    # fusion
    roi_sim_all = []
    for dl in range(1, 35):
        roi_sim = 1 - np.prod(1 - roi_conv_all[:, :dl], axis=1)
        roi_sim_all.append(roi_sim)
    roi_sim_all = np.array(roi_sim_all)

    # most similar regions
    top_regions = []
    top_regions_pre = np.argsort(-roi_sim_all, axis=1)[:, :21]
    for i in range(34):
        top_regions.append(
            np.append(
                roi,
                top_regions_pre[i][top_regions_pre[i] != roi][:20]
            )
        )
    top_regions = np.array(top_regions)

    # factor percentile significance
    pval, Z = top_region_sig(percents, top_regions)

    rows = (
        np.ones(top_regions.shape, dtype=int) *
        np.arange(top_regions.shape[0]).reshape(-1, 1)
    )

    return (top_regions, roi_sim_all[rows, top_regions],
            percents[top_regions], query_factor_order, target_factors, pval, Z)

In [333]:
t = time.time()

# factors the query region rank highest in
query_factor_order = np.argsort(-percents[roi])

print(time.time() - t)

# factors to search over based on detail level
target_factors = query_factor_order[
    np.isin(query_factor_order, included_factors)
]

print(time.time() - t)

# convolution
roi_percents = percents[roi, target_factors]
roi_conv_all = percents[:, target_factors] * roi_percents

print(time.time() - t)

# fusion
roi_sim_all = []
for dl in range(1, 35):
    roi_sim = 1 - np.prod(1 - roi_conv_all[:, :dl], axis=1)
    roi_sim_all.append(roi_sim)
roi_sim_all = np.array(roi_sim_all)

print(time.time() - t)

# most similar regions
top_regions = []
top_regions_pre = np.argsort(-roi_sim_all, axis=1)[:, :21]

print(time.time() - t)

for i in range(34):
    top_regions.append(
        np.append(
            roi,
            top_regions_pre[i][top_regions_pre[i] != roi][:20]
        )
    )
top_regions = np.array(top_regions)

print(time.time() - t)

# factor percentile significance
pval, Z = top_region_sig(percents, top_regions)

print(time.time() - t)

rows = (
    np.ones(top_regions.shape, dtype=int) *
    np.arange(top_regions.shape[0]).reshape(-1, 1)
)

print(time.time() - t)

0.0010929107666015625
0.001695394515991211
0.01774001121520996
0.062248945236206055
0.4551706314086914
0.4562246799468994
0.47256946563720703
0.4729955196380615


In [337]:
roi_sim_all.shape

(34, 88036)

In [323]:
def load_roi(genomic_location, gz_file_path):
    global roi
    if str.isalpha(str(genomic_location)[0]):
        # load roi
        gz_tb = tabix.open(gz_file_path)
        roi = np.array(
            list(gz_tb.querys(genomic_location))
        )[:, 3].astype(int)[0]
    else:
        roi = int(genomic_location)

        
def load_ranks(per_f):
    percents = np.load(per_f)
    return percents


def setup_included_factors(included_factors_str):
    global included_factors
    included_factors = np.int_(
        included_factors_str.strip('[').strip(']').split(',')
    )

def read_in_genomic_coordinates(regions_path):
    global regions_sep
    regions_sep = np.load(regions_path)

def find_most_similar_region_indices():
    global top_regions, roi_sim, roi_per, query_factor_order, target_factors, pval, Z
    (top_regions, roi_sim,
     roi_per, query_factor_order,
     target_factors, pval,
     Z) = simsearch(percents, roi, included_factors)

def setup_coordinates(top_regions):
    global top_region_coords
    top_region_coords = regions_sep[top_regions]

def setup_roiURLs(get_roiURL, top_regions, target_factors):
    global roiURLs
    roiURLs = []
    for dl in range(1, 35):
        if get_roiURL == 'True':
            # roiURLs.append('roiURL not available')
            roiURLs.append(roiURL(roi, top_region_coords[dl-1],
                                  dl, '_'.join(target_factors.astype(str))))
        else:
            roiURLs.append('roiURL not available')

def reorder_by_query_region():
    global return_roi_per, return_pval, return_Z, return_factor_order, return_metric_order, return_colors
    return_roi_per = roi_per[:, :, query_factor_order]
    return_factor_order = factor_order[query_factor_order]
    return_metric_order = metric_order[query_factor_order]
    return_colors = colors[query_factor_order]
    return_pval = pval[:, query_factor_order]
    return_Z = Z[:, query_factor_order]
    for i in range(len(return_factor_order)):
        return_factor_order[i] = return_factor_order[i].replace('_', '/')
        
def generate_result_array():
    global return_arr_full
    return_arr_full = []
    for dl in range(34):
        # for the roi, return results as json array
        # each element is an object of scores for each metric
        return_arr = []
        for rank in range(21):
            coord = top_region_coords[dl, rank]
            region_dict = {
                'rank': rank,
                'coordinates': coord[0] + ':' + coord[1] + '-' + coord[2],
                'region_ind': int(top_regions[dl, rank]),
                'score': float(roi_sim[dl, rank]),
                'percentiles': return_roi_per[dl, rank].tolist(),
                'factors': list(return_factor_order),
                'metrics': list(return_metric_order),
                'factor_colors': list(return_colors),
                #'sentence': sentences[dl, rank],
                'detail_level': dl + 1
            }
            if rank == 0:
                region_dict['query_factor_order'] = query_factor_order.tolist()
                region_dict['target_factors'] = target_factors.tolist()
                region_dict['top_region_factor_Z_score'] = return_Z[dl].tolist()
                region_dict['top_region_factor_pval'] = return_pval[dl].tolist()
                region_dict['roiURL'] = roiURLs[dl]
                #region_dict['factors'] = list(return_factor_order)
                #region_dict['metrics'] = list(return_metric_order)
                #region_dict['factor_colors'] = list(return_colors)
            return_arr.append(region_dict)

        return_arr_full.append(return_arr)
        


In [324]:
datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
per_f = datadir + 'regions_25kb_CT20230321.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.npy'


genomic_location = 'chr4:173514850-173514851'
gz_file_path = datadir + 'hg38.25kb.maxMean.CT20230118.sorted.rownum4.bed.gz'
# regions_path = datadir + 'hg38.25kb.maxMean.CT20230118.sorted.bed'
regions_path = datadir + 'hg38.25kb.maxMean.CT20230118.sorted.bed.npy'

get_roiURL = 'False'
included_factors_str = '[' + ','.join(list(np.arange(34).astype(str))) + ']'

In [336]:
t = time.time()

# load ROI
load_roi(genomic_location, gz_file_path)
print(time.time() - t)

# load region percentiles
percents = load_ranks(per_f) # new/optimized
print(time.time() - t)

# factors the user cares about
setup_included_factors(included_factors_str)
print(time.time() - t)

# read in genomic coordinates for each region
read_in_genomic_coordinates(regions_path)
print(time.time() - t)

# find most similar region inds
find_most_similar_region_indices()
print(time.time() - t)

# get coordinates
setup_coordinates(top_regions)
print(time.time() - t)

# dual track roiURL
setup_roiURLs(get_roiURL, top_regions, target_factors)
print(time.time() - t)

# reorder based on query region
reorder_by_query_region()
print(time.time() - t)

# generate result array
generate_result_array()
print(time.time() - t)

json_return = json.dumps(return_arr_full)
# sys.stdout.write(json_return)


print(time.time() - t)

0.02344965934753418
0.03974628448486328
0.04006218910217285
0.048032522201538086
0.5104649066925049
0.5109813213348389
0.5111782550811768
0.5114867687225342
0.5506551265716553
0.5824627876281738


In [ ]:
# return_arr_full